# Selecting Numerical and Categorical Features

Before training any model, define feature types deliberately.
This is a core modeling decision that affects preprocessing, model behavior, and evaluation quality.

## Why Feature Type Selection Matters

Incorrect type assignment can cause:
- Wrong encoding strategy
- Unnecessary or harmful scaling
- Lower model stability
- Harder interpretation
- Leakage risk if post-outcome features slip in

A column's storage dtype alone is not enough.
Use data meaning, prediction timing, and business logic.

In [ ]:
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

## Build a Simple Example Dataset

This dataset intentionally includes:
- true numerical columns
- true categorical columns
- binary flags
- an ID column to exclude
- a high-cardinality field to review

In [ ]:
rng = np.random.default_rng(42)
n = 300

df = pd.DataFrame({
    "CustomerID": [f"CUST_{i:05d}" for i in range(n)],
    "tenure": rng.integers(1, 72, n),
    "MonthlyCharges": rng.normal(70, 20, n).clip(15, 160),
    "TotalCharges": rng.normal(2500, 1200, n).clip(50, 12000),
    "gender": rng.choice(["Female", "Male"], n),
    "SeniorCitizen": rng.choice([0, 1], n, p=[0.83, 0.17]),
    "Contract": rng.choice(["Month-to-month", "One year", "Two year"], n, p=[0.58, 0.23, 0.19]),
    "PaymentMethod": rng.choice(["Electronic check", "Mailed check", "Bank transfer", "Credit card"], n),
    "InternetService": rng.choice(["DSL", "Fiber optic", "No"], n, p=[0.35, 0.5, 0.15]),
    "ZipCode": rng.integers(10000, 99999, n).astype(str),
})

# Synthetic binary target for demonstration
risk = (df["MonthlyCharges"] > 85).astype(int) + (df["Contract"] == "Month-to-month").astype(int)
prob = np.where(risk >= 1, 0.45, 0.12)
df["Churn"] = (rng.random(n) < prob).astype(int)

df.head()

## Step 1: Inspect Structure

Use `info`, `describe`, and unique-count checks.
Remember: storage type does not equal semantic type.

In [ ]:
display(df.info())
display(df.describe(include="all").T[["count", "nunique"]].head(12))

unique_counts = df.nunique().sort_values(ascending=False)
unique_counts

## Step 2: Define Target Early

Always separate target before feature type assignment to reduce leakage risk.

In [ ]:
TARGET_COLUMN = "Churn"
X = df.drop(columns=[TARGET_COLUMN])
y = df[TARGET_COLUMN]

print("X shape:", X.shape)
print("y shape:", y.shape)
print("Target present in X?", TARGET_COLUMN in X.columns)

## Step 3-5: Select Numerical, Categorical, and Excluded Columns

Use explicit lists, not implicit inference alone.
Document edge-case decisions (for example binary flags).

In [ ]:
# Explicit definition (recommended pattern)
NUMERICAL_FEATURES = [
    "tenure",
    "MonthlyCharges",
    "TotalCharges",
]

CATEGORICAL_FEATURES = [
    "gender",
    "SeniorCitizen",  # binary in storage, categorical in meaning
    "Contract",
    "PaymentMethod",
    "InternetService",
]

EXCLUDED_COLUMNS = [
    "CustomerID",   # identifier
    "ZipCode",      # high-cardinality categorical: defer until strategy chosen
]

ALL_FEATURES = NUMERICAL_FEATURES + CATEGORICAL_FEATURES

print("Numerical:", NUMERICAL_FEATURES)
print("Categorical:", CATEGORICAL_FEATURES)
print("Excluded:", EXCLUDED_COLUMNS)

## Validation Checks

Validate boundaries before modeling.

In [ ]:
def validate_feature_config(df: pd.DataFrame, target_col: str, numerical: list[str], categorical: list[str], excluded: list[str]) -> None:
    all_defined = numerical + categorical + excluded + [target_col]
    missing = [c for c in all_defined if c not in df.columns]
    if missing:
        raise ValueError(f"Missing columns in dataframe: {missing}")

    overlap_num_cat = set(numerical).intersection(categorical)
    overlap_feat_excl = set(numerical + categorical).intersection(excluded)

    if overlap_num_cat:
        raise ValueError(f"Columns overlap between numerical and categorical: {sorted(overlap_num_cat)}")
    if overlap_feat_excl:
        raise ValueError(f"Columns overlap between features and excluded: {sorted(overlap_feat_excl)}")
    if target_col in numerical or target_col in categorical:
        raise ValueError("Target column must not be included in features.")

    undefined = set(df.columns) - set(all_defined)
    if undefined:
        print("Warning: These columns are not assigned to any group:", sorted(undefined))

    print("Feature configuration validation passed.")

validate_feature_config(df, TARGET_COLUMN, NUMERICAL_FEATURES, CATEGORICAL_FEATURES, EXCLUDED_COLUMNS)

## Build Type-Aware Preprocessing

Numerical and categorical columns should flow through different preprocessing branches.

In [ ]:
numeric_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

categorical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore")),
])

preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_pipeline, NUMERICAL_FEATURES),
    ("cat", categorical_pipeline, CATEGORICAL_FEATURES),
])

X_model = df[ALL_FEATURES].copy()
Xt = preprocessor.fit_transform(X_model)

print("Transformed matrix type:", type(Xt))
print("Transformed shape:", Xt.shape)

## Edge Cases and Decisions

1. Binary 0/1 columns
- May be stored numeric but conceptually categorical.
- Keep decision consistent and documented.

2. High-cardinality categories
- One-hot may create too many sparse columns.
- Consider frequency encoding, grouping, or target encoding (with strict leakage controls).

3. Datetime columns
- Avoid raw timestamps directly.
- Derive features such as day-of-week, month, hour, or durations.

In [ ]:
def prediction_moment_test(feature_name: str, available_at_prediction_time: bool) -> str:
    if not available_at_prediction_time:
        return f"{feature_name}: REJECT (not available at prediction time)"
    return f"{feature_name}: ACCEPT"

print(prediction_moment_test("tenure", True))
print(prediction_moment_test("CancellationReason", False))

## Final Checklist

Before modeling, verify:
- Target column is clearly defined
- Numerical and categorical lists are explicit
- No overlap between target and features
- IDs and post-outcome columns are excluded
- High-cardinality handling is intentional
- All features pass prediction-time availability
- Decisions are documented for reproducibility

## Closing Reflection

Selecting feature types is a conceptual modeling choice, not a mechanical task.
Correct categorization leads to stable preprocessing, honest evaluation, and more reliable production behavior.